In [1]:
#dataset_1

In [2]:
#Data Loading & Initial Inspection

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import make_column_transformer

# Load dataset
df = pd.read_csv('leaves_training_final.csv')

# Initial inspection
print(f"Dataset shape: {df.shape}")
print("\nFirst 5 rows:")
print(df.head())
print("\nData types and missing values:")
print(df.info())
print("\nDescriptive statistics:")
print(df.describe())

In [ ]:
#Species Distribution Analysis

In [ ]:
# Count examples per species
species_counts = df['name'].value_counts()

print("\nNumber of examples per species:")
print(species_counts)

# Visualization
plt.figure(figsize=(12,6))
species_counts.plot(kind='bar', color='green')
plt.title('Number of Samples per Plant Species', fontsize=14)
plt.xlabel('Species', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.xticks(rotation=90)
plt.grid(axis='y', linestyle='--')
plt.show()

In [ ]:
Feature Consistency Analysis

In [ ]:
# Histograms for numerical features
fig, axes = plt.subplots(1, 2, figsize=(12,5))
df['length'].hist(ax=axes[0], bins=20, color='teal')
axes[0].set_title('Leaf Length Distribution')
axes[0].set_xlabel('Length (cm)')

df['width'].hist(ax=axes[1], bins=20, color='olive')
axes[1].set_title('Leaf Width Distribution')
axes[1].set_xlabel('Width (cm)')
plt.tight_layout()
plt.show()

In [ ]:
# Length vs Width by species
plt.figure(figsize=(14,8))
sns.scatterplot(data=df, x='length', y='width', hue='name', 
               palette='tab20', s=100, alpha=0.8)
plt.title('Leaf Dimensions by Species', fontsize=14)
plt.xlabel('Length (cm)', fontsize=12)
plt.ylabel('Width (cm)', fontsize=12)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, linestyle='--')
plt.show()

# Boxplots of length by species
plt.figure(figsize=(14,6))
sns.boxplot(data=df, x='name', y='length', palette='Set3')
plt.title('Leaf Length Distribution by Species', fontsize=14)
plt.xlabel('Species', fontsize=12)
plt.ylabel('Length (cm)', fontsize=12)
plt.xticks(rotation=90)
plt.grid(axis='y', linestyle='--')
plt.show()

In [ ]:
Width Prediction Modeling

In [ ]:
# Feature engineering
X = df[['name', 'length']]  # Using species and length as predictors
y = df['width']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

# Preprocessing pipeline
preprocessor = make_column_transformer(
    (OneHotEncoder(), ['name']),  # One-hot encode species
    remainder='passthrough'       # Keep length as-is
)

In [ ]:
# Initialize models
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(random_state=42)
}

# Train and evaluate
results = {}
for name, model in models.items():
    # Create pipeline
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])
    
    # Train
    pipe.fit(X_train, y_train)
    
    # Predict
    y_pred = pipe.predict(X_test)
    
    # Evaluate
    rmse = mean_squared_error(y_test, y_pred, squared=False)
    r2 = r2_score(y_test, y_pred)
    
    results[name] = {'RMSE': rmse, 'R²': r2}
    
    # Print feature importance for Random Forest
    if name == 'Random Forest':
        # Get feature names after one-hot encoding
        ohe = pipe.named_steps['preprocessor'].named_transformers_['onehotencoder']
        species_features = ohe.get_feature_names_out(['name'])
        all_features = list(species_features) + ['length']
        
        # Get importances
        importances = pipe.named_steps['regressor'].feature_importances_
        feat_imp = pd.Series(importances, index=all_features).sort_values(ascending=False)
        
        print("\nRandom Forest Feature Importances:")
        print(feat_imp.head(10))

# Display results
results_df = pd.DataFrame(results).T
print("\nModel Performance Comparison:")
print(results_df)

In [ ]:
#dataset_2

In [ ]:
#Data_Cleaning

In [ ]:
import pandas as pd
import numpy as np

# Load the data
df = pd.read_csv('DAVWorkshop_Survey.csv')

# Extract distance and time columns
distance = df['Approx. Distance of your home from IIT Bhilai (in Kms). Enter only a number (floating point).']
time = df['Approx. Time to reach IIT Bhilai from your hometown (in hours). Enter only a number (floating point)']

In [ ]:
Outlier Detection and Treatment

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Visual inspection with boxplots
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.boxplot(x=distance_clean)
plt.title('Distance Outliers')

plt.subplot(1, 2, 2)
sns.boxplot(x=time_clean)
plt.title('Time Outliers')
plt.show()

# Statistical outlier detection using IQR
def remove_outliers_iqr(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return series[(series >= lower_bound) & (series <= upper_bound)]

distance_no_outliers = remove_outliers_iqr(distance_clean)
time_no_outliers = remove_outliers_iqr(time_clean)

In [ ]:
#Regression_Model

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

# Prepare data
X = time_no_outliers.values.reshape(-1, 1)  # Time as input
y = distance_no_outliers.values            # Distance as output

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train model
model = LinearRegression()
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Model Evaluation:")
print(f"MAE: {mae:.2f} km")
print(f"MSE: {mse:.2f} km²")
print(f"R² Score: {r2:.2f}")

# Visualize results
plt.figure(figsize=(10, 6))
plt.scatter(X_test, y_test, color='blue', label='Actual')
plt.plot(X_test, y_pred, color='red', linewidth=2, label='Predicted')
plt.xlabel('Time (hours)')
plt.ylabel('Distance (km)')
plt.title('Distance vs Time Regression')
plt.legend()
plt.show()